In [1]:
import pandas as pd
import sqlite3

# 连接 SQLite 数据库
conn_info = sqlite3.connect("../query_dataset/indexInfo_query.db")

# 读取 index_info 表
df_info = pd.read_sql("SELECT * FROM index_info", conn_info)

# 示例输出
print(df_info.head())

conn_info.close()


                   Exchange Currency
0   New York Stock Exchange      USD
1                    NASDAQ      USD
2  Hong Kong Stock Exchange      HKD
3   Shanghai Stock Exchange      CNY
4      Tokyo Stock Exchange      JPY


In [2]:
import duckdb

# 连接 DuckDB 数据库
con_trade = duckdb.connect("../query_dataset/indexTrade_query.db")

# 读取 index_trade 表
df_trade = con_trade.execute("SELECT * FROM index_trade").fetchdf()

# 示例输出
print(df_trade.head())

con_trade.close()


  Index                          Date         Open         High          Low  \
0   HSI            31 Dec 1986, 00:00  2568.300049  2568.300049  2568.300049   
1   HSI  January 02, 1987 at 12:00 AM  2540.100098  2540.100098  2540.100098   
2   HSI           1987-01-05 00:00:00  2552.399902  2552.399902  2552.399902   
3   HSI            06 Jan 1987, 00:00  2583.899902  2583.899902  2583.899902   
4   HSI            07 Jan 1987, 00:00  2607.100098  2607.100098  2607.100098   

         Close    Adj Close    CloseUSD  
0  2568.300049  2568.300049  333.879006  
1  2540.100098  2540.100098  330.213013  
2  2552.399902  2552.399902  331.811987  
3  2583.899902  2583.899902  335.906987  
4  2607.100098  2607.100098  338.923013  


In [9]:
import sqlite3
import pandas as pd
from openai import AzureOpenAI
from tqdm import tqdm

# ========= Step 1: Load index_info ==========
conn = sqlite3.connect("../query_dataset/indexInfo_query.db")
df_info = pd.read_sql("SELECT * FROM index_info", conn)
conn.close()

# ========= Step 2: OpenAI API Setup ==========
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)
deployment_name = "gpt-4o" 

# ========= Step 3: Define LLM inference function ==========
def infer_region_and_symbol(exchange_name, client, deployment_name):
    prompt = (
         f"""You are given the name of a stock exchange: '{exchange_name}'.
            Please answer the following two fields:
            - Region: The global region it belongs to (e.g., Asia, Europe, North America).
            - Index Symbol: The exact symbol used in financial datasets or APIs (e.g., from Yahoo Finance, TradingView, Bloomberg).

            Important:
            - The Index Symbol must be the actual code used in trading data (e.g., in Yahoo Finance).
            - Do NOT return descriptive names like “KOSPI” or “JSE All Share Index”.
            - Return only the standard **index code** used in market data feeds.

            Format:
            Region: <region>  
            Index Symbol: <symbol>"""
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a financial assistant that maps stock exchanges to their global region and index symbol."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=50
        )
        output = response.choices[0].message.content.strip().split("\n")
        region = output[0].replace("Region:", "").strip()
        index_symbol = output[1].replace("Index Symbol:", "").strip()
        return region, index_symbol
    except Exception as e:
        print(f"[ERROR] {exchange_name}: {e}")
        return None, None

# ========= Step 4: Apply to each row with print ==========
df_info["region"] = ""
df_info["index_symbol"] = ""

for i, row in tqdm(df_info.iterrows(), total=len(df_info)):
    exchange = row.get("Exchange", "")
    if not exchange:
        continue
    region, symbol = infer_region_and_symbol(exchange, client, deployment_name)
    df_info.at[i, "region"] = region
    df_info.at[i, "index_symbol"] = symbol
    print(f"[{i}] Exchange: {exchange} | Region: {region} | Symbol: {symbol}")


  0%|          | 0/14 [00:00<?, ?it/s]

  7%|▋         | 1/14 [00:00<00:07,  1.68it/s]

[0] Exchange: New York Stock Exchange | Region: North America | Symbol: ^NYA


 14%|█▍        | 2/14 [00:00<00:05,  2.34it/s]

[1] Exchange: NASDAQ | Region: North America | Symbol: IXIC


...

[11] Exchange: SIX Swiss Exchange | Region: Europe | Symbol: SMI


 93%|█████████▎| 13/14 [00:05<00:00,  2.39it/s]

[12] Exchange: Taiwan Stock Exchange | Region: Asia | Symbol: TWII


100%|██████████| 14/14 [00:05<00:00,  2.41it/s]

[13] Exchange: Johannesburg Stock Exchange | Region: Africa | Symbol: JSE


In [10]:
df_info

,Exchange,Currency,region,index_symbol
0,New York Stock Exchange,USD,North America,^NYA
1,NASDAQ,USD,North America,IXIC
2,Hong Kong Stock Exchange,HKD,Asia,^HSI
3,Shanghai Stock Exchange,CNY,Asia,000001.SS
4,Tokyo Stock Exchange,JPY,Asia,^N225
5,Euronext,EUR,Europe,^N100
6,Shenzhen Stock Exchange,CNY,Asia,399001.SZ
7,Toronto Stock Exchange,CAD,North America,^GSPTSE
8,National Stock Exchange of India,INR,Asia,NIFTY 50
9,Frankfurt Stock Exchange,EUR,Europe,^GDAXI


In [13]:
import duckdb
import pandas as pd
import openai
import json

import duckdb
import pandas as pd
import openai
import json
import re
import ast

# ========= Step 1: 加载 DuckDB 中的交易数据 ==========
con = duckdb.connect("../query_dataset/indexTrade_query.db")
df_trade = con.execute("SELECT * FROM index_trade").fetchdf()
con.close()
# ========= Step 4: 调用 OpenAI GPT ==========
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)
deployment_name = "gpt-4o" 

# ========= Step 2: 提取唯一索引候选 ==========
index_symbol_list = sorted(df_info["index_symbol"].dropna().unique().tolist())
index_list = sorted(df_trade["Index"].dropna().unique().tolist())

# ========= Step 3: 构造 GPT Prompt ==========
def build_mapping_prompt(index_symbols, index_list):
    return (
        "You are given two lists of stock index codes from two different datasets.\n\n"
        f"List A (`index_symbol` from metadata):\n{json.dumps(index_symbols, indent=2)}\n\n"
        f"List B (`Index` from trading data):\n{json.dumps(index_list, indent=2)}\n\n"
        "Each item in List A corresponds to exactly one semantically equivalent item in List B.\n"
        "Your job is to map List A to List B.\n\n"
        "Respond ONLY with a Python dictionary in the following format (no explanation, no markdown):\n"
        "{\n"
        "  '<index_symbol_from_list_A>': '<corresponding_Index_from_list_B>',\n"
        "  ...\n"
        "}"
    )

prompt = build_mapping_prompt(index_symbol_list, index_list)

# ========= Step 4: 调用 GPT 获取映射 ==========

response = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", "content": "You are a financial data assistant matching index symbols across datasets."},
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=800
)

# ========= Step 5: 解析 GPT 返回的字典 ==========
reply = response.choices[0].message.content.strip()

try:
    # 兼容 GPT 返回可能带 markdown code block 的格式
    match = re.search(r"\{.*\}", reply, re.DOTALL)
    code_str = match.group(0) if match else reply
    index_symbol_to_index = ast.literal_eval(code_str)
except Exception as e:
    print("❌ Failed to parse GPT response:\n", reply)
    raise e

print("\n✅ Inferred Mapping:")
print(index_symbol_to_index)

# ========= Step 6: 应用映射并 Join ==========
df_info["index_mapped"] = df_info["index_symbol"].map(index_symbol_to_index)

df_joined = pd.merge(df_info, df_trade, left_on="index_mapped", right_on="Index", how="inner")

# ========= Step 7: 输出示例 ==========
print("\n✅ Sample joined result:")
print(df_joined[["Exchange", "index_symbol", "Index", "region"]].head())




✅ Inferred Mapping:
{'000001.SS': '000001.SS', '399001.SZ': '399001.SZ', 'IXIC': 'IXIC', 'JSE': 'J203.JO', 'KS11': 'NSEI', 'NIFTY 50': 'NSEI', 'SMI': 'SSMI', 'TWII': 'TWII', '^GDAXI': 'GDAXI', '^GSPTSE': 'GSPTSE', '^HSI': 'HSI', '^N100': 'N100', '^N225': 'N225', '^NYA': 'NYA'}

✅ Sample joined result:
                  Exchange index_symbol Index         region
0  New York Stock Exchange         ^NYA   NYA  North America
1  New York Stock Exchange         ^NYA   NYA  North America
2  New York Stock Exchange         ^NYA   NYA  North America
3  New York Stock Exchange         ^NYA   NYA  North America
4  New York Stock Exchange         ^NYA   NYA  North America


In [14]:
df_joined

,Exchange,Currency,region,index_symbol,index_mapped,Index,Date,Open,High,Low,Close,Adj Close,CloseUSD
0,New York Stock Exchange,USD,North America,^NYA,NYA,NYA,"December 31, 1965 at 12:00 AM",528.690002,528.690002,528.690002,528.690002,528.690002,528.690002
1,New York Stock Exchange,USD,North America,^NYA,NYA,NYA,"03 Jan 1966, 00:00",527.210022,527.210022,527.210022,527.210022,527.210022,527.210022
2,New York Stock Exchange,USD,North America,^NYA,NYA,NYA,1966-01-04 00:00:00,527.840027,527.840027,527.840027,527.840027,527.840027,527.840027
3,New York Stock Exchange,USD,North America,^NYA,NYA,NYA,"05 Jan 1966, 00:00",531.119995,531.119995,531.119995,531.119995,531.119995,531.119995
4,New York Stock Exchange,USD,North America,^NYA,NYA,NYA,"January 06, 1966 at 12:00 AM",532.070007,532.070007,532.070007,532.070007,532.070007,532.070007
...,...,...,...,...,...,...,...,...,...,...,...,...,...
107565,Johannesburg Stock Exchange,ZAR,Africa,JSE,J203.JO,J203.JO,2021-05-25 00:00:00,66054.921880,66812.453130,66022.976560,66076.679690,66076.679690,4625.367578
107566,Johannesburg Stock Exchange,ZAR,Africa,JSE,J203.JO,J203.JO,2021-05-26 00:00:00,66076.679690,66446.367190,66030.351560,66108.226560,66108.226560,4627.575859
107567,Johannesburg Stock Exchange,ZAR,Africa,JSE,J203.JO,J203.JO,"27 May 2021, 00:00",66108.226560,66940.250000,66102.546880,66940.250000,66940.250000,4685.817500
107568,Johannesburg Stock Exchange,ZAR,Africa,JSE,J203.JO,J203.JO,2021-05-28 00:00:00,66940.250000,67726.562500,66794.609380,67554.859380,67554.859380,4728.840157


In [22]:
import pandas as pd
import openai
import json
import ast
import re
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)
deployment_name = "gpt-4o-mini" 

# ==== 假设 df_asia 已经准备好 ====
df_asia = df_joined[df_joined["region"] == "Asia"].copy()
date_list = df_asia["Date"].tolist()

# ==== GPT 批处理函数 ====
def normalize_batch_dates(dates):
    prompt = (
    "You are given a list of date strings in natural language format.\n"
    "Convert each of them into ISO format (yyyy-mm-dd).\n"
    "If any date is invalid, return 'None' for that entry.\n\n"
    "Input:\n"
    + json.dumps(dates, indent=2) + "\n\n"
    "Output:\n"
    "**Only return a plain Python list of ISO-formatted date strings in same order**, no explanation, no variable assignments, and no markdown (no ```python)."
)

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You convert a list of messy human-readable dates into ISO 8601 format."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=500
        )
        text = response.choices[0].message.content.strip()
        match = re.search(r"\[(.*?)\]", text, re.DOTALL)
        if match:
            parsed_list = ast.literal_eval("[" + match.group(1) + "]")
        else:
            parsed_list = ast.literal_eval(text)
        return parsed_list
    except Exception as e:
        print(f"❌ GPT error on batch: {e}")
        print("Raw reply:\n", response.choices[0].message.content if 'response' in locals() else "")
        return [None] * len(dates)

# ==== 多线程调用 ====
from concurrent.futures import ThreadPoolExecutor, as_completed

def parallel_date_normalization(date_list, batch_size=50, max_workers=32):
    batches = [date_list[i:i + batch_size] for i in range(0, len(date_list), batch_size)]
    normalized = [None] * len(date_list)
    failed_batches = {}  # {batch_index: batch_data}

    # === 并发主调用 ===
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {}
        for idx, batch in enumerate(batches):
            future = executor.submit(normalize_batch_dates, batch)
            futures[future] = idx

        for future in tqdm(as_completed(futures), total=len(futures)):
            idx = futures[future]
            try:
                result = future.result()
                normalized[idx * batch_size : idx * batch_size + len(result)] = result
            except Exception as e:
                print(f"❌ Batch {idx} failed during main thread: {e}")
                failed_batches[idx] = batch

    # === 后处理失败 batch：统一 retry ===
    if failed_batches:
        print(f"\n🔁 Retrying {len(failed_batches)} failed batches (after main run)...")
        for idx, batch in failed_batches.items():
            try:
                result = normalize_batch_dates(batch)
                normalized[idx * batch_size : idx * batch_size + len(result)] = result
                print(f"✅ Retry batch {idx} succeeded")
            except Exception as e:
                print(f"❌ Retry batch {idx} failed again: {e}")
                # 可选写入日志：log_failed_batch(idx, batch)
                continue

    return normalized

# ==== 执行并发解析 ====
standardized_dates = parallel_date_normalization(date_list)

# ==== 写入结果 ====
df_asia["parsed_date"] = pd.to_datetime(standardized_dates, errors="coerce")
df_asia

100%|██████████| 930/930 [04:19<00:00,  3.58it/s]


,Exchange,Currency,region,index_symbol,index_mapped,Index,Date,Open,High,Low,Close,Adj Close,CloseUSD,parsed_date
26637,Hong Kong Stock Exchange,HKD,Asia,^HSI,HSI,HSI,"31 Dec 1986, 00:00",2568.300049,2568.300049,2568.300049,2568.300049,2568.300049,333.879006,1986-12-31
26638,Hong Kong Stock Exchange,HKD,Asia,^HSI,HSI,HSI,"January 02, 1987 at 12:00 AM",2540.100098,2540.100098,2540.100098,2540.100098,2540.100098,330.213013,1987-01-02
26639,Hong Kong Stock Exchange,HKD,Asia,^HSI,HSI,HSI,1987-01-05 00:00:00,2552.399902,2552.399902,2552.399902,2552.399902,2552.399902,331.811987,1987-01-05
26640,Hong Kong Stock Exchange,HKD,Asia,^HSI,HSI,HSI,"06 Jan 1987, 00:00",2583.899902,2583.899902,2583.899902,2583.899902,2583.899902,335.906987,1987-01-06
26641,Hong Kong Stock Exchange,HKD,Asia,^HSI,HSI,HSI,"07 Jan 1987, 00:00",2607.100098,2607.100098,2607.100098,2607.100098,2607.100098,338.923013,1987-01-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105219,Taiwan Stock Exchange,TWD,Asia,TWII,TWII,TWII,"May 25, 2021 at 12:00 AM",16444.750000,16657.599610,16444.750000,16595.669920,16595.669920,663.826797,2021-05-25
105220,Taiwan Stock Exchange,TWD,Asia,TWII,TWII,TWII,2021-05-26 00:00:00,16645.169920,16706.289060,16523.230470,16643.689450,16643.689450,665.747578,2021-05-26
105221,Taiwan Stock Exchange,TWD,Asia,TWII,TWII,TWII,"27 May 2021, 00:00",16591.699220,16601.609380,16419.419920,16601.609380,16601.609380,664.064375,2021-05-27
105222,Taiwan Stock Exchange,TWD,Asia,TWII,TWII,TWII,"28 May 2021, 00:00",16690.039060,16889.009770,16690.039060,16870.859380,16870.859380,674.834375,2021-05-28


In [23]:
# ==== Step 1: 过滤 2020年及之后的数据 ====
df_filtered = df_asia[df_asia["parsed_date"].dt.year >= 2020].copy()

# ==== Step 2: 计算每行 intraday volatility ====
df_filtered["volatility"] = (df_filtered["High"] - df_filtered["Low"]) / df_filtered["Open"]

# ==== Step 3: 分组取均值 ====
vol_summary = (
    df_filtered
    .groupby("index_symbol")["volatility"]
    .mean()
    .reset_index()
    .sort_values(by="volatility", ascending=False)
)

# ==== Step 4: 打印结果 ====
print("\n📊 Top Asian indices by average intraday volatility (since 2020):")
print(vol_summary.head(10))



📊 Top Asian indices by average intraday volatility (since 2020):
  index_symbol  volatility
1    399001.SZ    0.018341
2         KS11    0.017072
3     NIFTY 50    0.017072
5         ^HSI    0.014932
0    000001.SS    0.013781
6        ^N225    0.013427
4         TWII    0.013220
